# Empirical coverage of the five CI methods

A confidence interval *says* it covers the true value 95 % of the time, but
does it? We check directly via Monte Carlo simulation on a synthetic data
generating process (DGP) where the true uplift of every mined rule is known
in closed form.

For each combination of sample size `n` and CI method we:

1. Generate `N_REPLICATES` fresh datasets of size `n` from the DGP.
2. Mine action rules on each replicate.
3. Compute the CI with each method.
4. Record whether each interval contains the *analytic true* uplift of that
   rule under the DGP.

**Empirical coverage** is the fraction of intervals that contain their target;
a well-calibrated method should hit ~0.95.

**Outputs**:
- `notebooks/inference_studies/results/coverage_simulation.csv` —
  one row per (n × method): empirical coverage, mean width, mean runtime.
- `notebooks/inference_studies/results/coverage_simulation_records.csv` —
  one row per (replicate × n × method × rule): hit/miss + width.

The synthetic DGP and the runner live in `tests/simulation/coverage_simulation.py`
so the same code path is used both in the test suite and here.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
# `tests/simulation/coverage_simulation.py` ships with the package source tree.
sys.path.insert(0, str(REPO_ROOT))
from tests.simulation.coverage_simulation import (  # noqa: E402
    DGPParams,
    generate_dataset,
    run_grid,
    true_uplift,
)

import time

OUT_SUMMARY = RESULTS_DIR / 'coverage_simulation.csv'
OUT_RECORDS = RESULTS_DIR / 'coverage_simulation_records.csv'


## Parameters

We default to **smoke-scale** so a clean-kernel *Run All* completes in a few
minutes. Bump the constants to recover the article-grade numbers (parameters
are commented next to each line).


In [ ]:
SAMPLE_SIZES = [200, 500, 2000]    # article-grade: [100, 500, 2000, 10000]
N_REPLICATES = 30                  # article-grade: 500
N_BOOTSTRAP = 200                  # article-grade: 500
N_MC = 2000                        # article-grade: 10_000
BASE_SEED = 20260524

params = DGPParams()
print(f'sample sizes: {SAMPLE_SIZES}')
print(f'replicates:   {N_REPLICATES}')
print(f'bootstrap B:  {N_BOOTSTRAP}')
print(f'Bayesian M:   {N_MC}')


## DGP sanity check

A quick view of the synthetic dataset and one rule's analytic true uplift to
confirm the DGP plumbing is sane.


In [ ]:
from action_rules import ActionRules

df_demo = generate_dataset(2000, params, seed=BASE_SEED)
print('Y prevalence:', df_demo['Y'].value_counts(normalize=True).round(3).to_dict())

ar_demo = ActionRules(
    min_stable_attributes=1,
    min_flexible_attributes=1,
    min_undesired_support=20,
    min_desired_support=20,
    min_undesired_confidence=0.5,
    min_desired_confidence=0.5,
)
ar_demo.fit(
    df_demo,
    stable_attributes=['S1', 'S2'],
    flexible_attributes=['F1', 'F2'],
    target='Y',
    target_undesired_state='0',
    target_desired_state='1',
)
print(f'rules mined: {len(ar_demo.output.action_rules)}')
rule0 = ar_demo.output.action_rules[0]
print(
    f"rule 0: sample uplift = {rule0['uplift']:.4f}; "
    f"analytic true uplift = {true_uplift(rule0, ar_demo.output.column_values, params):.4f}"
)


## Run the simulation grid

Each replicate generates a fresh dataset, mines rules, computes CIs with all
five methods, and records whether each interval contains the true uplift.


In [ ]:
t0 = time.perf_counter()
records_df, summary = run_grid(
    SAMPLE_SIZES,
    n_replicates=N_REPLICATES,
    params=params,
    n_bootstrap=N_BOOTSTRAP,
    n_mc=N_MC,
    base_seed=BASE_SEED,
    progress=False,
)
elapsed = time.perf_counter() - t0
print(f'simulation finished in {elapsed/60:.1f} min')
print(f'records: {len(records_df):,} rows; summary: {len(summary)} (n x method) cells')


## Empirical coverage, width, and runtime

Three pivots tell the full story:

- **Coverage**: how often the interval contains the truth. Good methods sit
  near 0.95; under-coverage means the interval is too narrow.
- **Width**: how informative the interval is. Should shrink ~ 1/√n.
- **Runtime**: bootstrap is the slowest; analytic is essentially free.


In [ ]:
pivot_cov = summary.pivot(index='n', columns='method', values='empirical_coverage').round(3)
pivot_width = summary.pivot(index='n', columns='method', values='mean_width').round(4)
pivot_time = summary.pivot(index='n', columns='method', values='mean_runtime_s').round(3)

print('Empirical 95% coverage:')
print(pivot_cov.to_string())
print('\nMean interval width:')
print(pivot_width.to_string())
print('\nMean wall-clock per replicate (seconds):')
print(pivot_time.to_string())


## Persist results

Both CSVs are saved so they can be inspected or fed into a downstream plotting
step without rerunning the simulation.


In [ ]:
summary.to_csv(OUT_SUMMARY, index=False)
records_df.to_csv(OUT_RECORDS, index=False)
print(f'wrote {OUT_SUMMARY.relative_to(REPO_ROOT)} ({OUT_SUMMARY.stat().st_size:,} bytes)')
print(f'wrote {OUT_RECORDS.relative_to(REPO_ROOT)} ({OUT_RECORDS.stat().st_size:,} bytes)')


## Interactive plot

Quick view only — re-runs ignore the saved CSV. Matplotlib is an optional
dependency, so we guard the import.


In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    for method, sub in summary.groupby('method'):
        ax.plot(sub['n'], sub['empirical_coverage'], marker='o', label=method)
    ax.axhline(0.95, color='gray', linestyle='--', linewidth=1, label='nominal 0.95')
    ax.set_xscale('log')
    ax.set_xlabel('sample size n (log scale)')
    ax.set_ylabel('empirical 95% coverage')
    ax.set_title('Coverage calibration')
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed; skip plot')


## Reading the table

- **Wald** tends to under-cover at small `n` (the normal approximation
  underestimates variance for proportions near 0 or 1).
- **Wilson** and **Bayesian** are well-calibrated even at `n = 200`.
- **Bootstrap percentile** and **BCa** converge to nominal as `n` grows;
  BCa is slightly more conservative.

Width should shrink as `1/√n`. Doubling `N_REPLICATES` halves the Monte Carlo
noise on the coverage estimate but leaves the mean width unchanged.
